In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

In [ ]:
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X, y_train = load_iris(return_X_y=True)

scaler = StandardScaler()
X_norm = scaler.fit_transform(X_train, y_train)

X_train, X_test, y_train, y_test = train_test_split(X_norm, y, test_size=0.2, random_state=100)

In [ ]:
iris_data = pd.DataFrame(np.column_stack((X_train, y)))
iris_data.info()
iris_data.describe().loc[["min", "max", "mean", "std"]]

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       150 non-null    float64
 1   1       150 non-null    float64
 2   2       150 non-null    float64
 3   3       150 non-null    float64
 4   4       150 non-null    float64
dtypes: float64(5)
memory usage: 6.0 KB


,0,1,2,3,4
min,4.300000,2.000000,1.000000,0.100000,0.000000
max,7.900000,4.400000,6.900000,2.500000,2.000000
mean,5.843333,3.057333,3.758000,1.199333,1.000000
std,0.828066,0.435866,1.765298,0.762238,0.819232


In [62]:
# utils
from scipy.stats import entropy
from collections import Counter
import random

def majority_vote(classes):
    ctr = Counter(classes)
    highest_count = ctr.most_common()[0][1]
    modes = [item for item, count in ctr.items() if count == highest_count]
    prediction = random.choice(modes)
    return prediction

In [ ]:
class DecisionTreeNode:
    """
        A class to represent a decision tree node
        
        Attributes:
            lc   (DecisionTreeNode):        left child
            rc   (DecisionTreeNode):        right child
            
            X_train    (numpy.ndarray):     training examples at current node
            y_train    (numpy.ndarray):     training lables at current node
            size       (int):               datasetsize
            
            split_feat (int):               number of feature to split
            split_thrs (float):             threshold to split on
            
            ig   (float):                   information gain
            gini (float):                   gini impurity
    """
    
    def __init__(self, X_train, y_train):
        if X_train.shape[0] != y_train.shape[0]:
            raise ValueError("X_train and y_train must have the same number of rows")
        self.X_train = X_train
        self.y_train = y_train
        self.size = X_train.shape[0]
    
        self.lc = None
        self.rc = None
        
        self.gini = -1
        self.ig = -1
    
    def __calc_gini(self, p):
        return 1 - np.sum(p ** 2)
    
    def depth(self) -> int:
        """
        Calculates and returns the maximum depth of the tree rooted at this node.
        
        Returns:
            int: The maximum depth of the tree (0 for leaf nodes)
        """
        if self.lc is None and self.rc is None:
            # Leaf node - depth is 0
            return 0
        
        left_depth = self.lc.depth() if self.lc is not None else 0
        right_depth = self.rc.depth() if self.rc is not None else 0
        
        return 1 + max(left_depth, right_depth)

    def count_nodes(self) -> int:
        """
        Counts and returns the total number of nodes in the tree rooted at this node.
        
        Returns:
            int: Total number of nodes in the subtree
        """
        # Count this node
        count = 1
        
        # Add nodes from left subtree
        if self.lc is not None:
            count += self.lc.count_nodes()
        
        # Add nodes from right subtree
        if self.rc is not None:
            count += self.rc.count_nodes()
        
        return count
        
    def find_split(self, split_method: str="ig", update_split_criteria: bool=True) -> tuple[int, float]:
        """
        Determines and sets the feature and threshold pair that results in the best split

        Returns:
            (int, int): feature and threshold to split    
        """
        if not X_train or not y_train:
            raise ValueError("Cannot split node with no data X_train, y_train")
        
        # partitions the data to two subsets based on the best split
        m, n = X_train.shape
        
        e = entropy(y_train)
        
        best_gini = -1
        best_ig = -1
        best_feat = 0
        best_thrs = 0
        
        for j in range(n):
            # sort the data with respect to the jth feature
            sorted_indices = np.argsort(X_train[:,j])
            sorted_labels = y_train[sorted_indices]
            for i in range(m-1):
                # determine the threshold to split data on
                t = np.mean(X_train[i, j], X_train[i+1, j])
                
                y_left = y_train[X_train[j] <= t]
                y_right = y_train[X_train[j] > t]
                
                # calculate the split criteria
                if split_method.lower() == "gini":
                    gini = self.__calc_gini(np.array([y_left, y_right]))
                    if gini > best_gini:
                        best_gini = gini
                        best_feat = j
                        best_thrs = t
                elif split_method.lower() in ["ig", "information gain", "gain"]:
                    ig = e - ((y_left.shape[0] / y_train.shape[0]) * entropy(y_left) + (y_right.shape[0] / y_train.shape[0]) * entropy(y_right))
                    if ig > best_ig:
                        best_ig = ig
                        best_feat = j
                        best_thrs = t
                else:
                    raise ValueError("Invalid split method")
            
        if update_split_criteria:
            self.ig = best_ig
            self.gini = best_gini
            
        self.split_feat = best_feat
        self.split_thrs = best_thrs
        
        return (self.split_feat, self.split_thrs)
    
    def split(self) -> tuple[DecisionTreeNode, DecisionTreeNode]:
        """
        Partitions the node according to the best split recursively until a stopping condition is reached
        
        Returns:
            (DecisionTreeNode, DecisionTreeNode): left and right children after split
        """
        
        
        if not self.split_feat or not self.split_thrs:
            self.find_split()     
        
        X_left = X_train[X_train[self.split_feat] <= self.split_thrs]
        y_left = y_train[X_train[self.split_feat] <= self.split_thrs]
        X_right = X_train[X_train[self.split_feat] > self.split_thrs]
        y_right = y_train[X_train[self.split_feat] > self.split_thrs]
        
        self.lc = DecisionTreeNode(X_left, y_left)
        self.rc = DecisionTreeNode(X_right, y_right)
        
        return (self.lc, self.rc)

In [ ]:
class DecisionTreeClassifier:
    """
        Represents a decision tree classification model with standard fit and predict API
        
        Attributes:
            root     (DecisionTreeNode): decision tree root node 
            
            X_train  (numpy.ndarray):    training examples
            y_train  (numpy.ndarray):    training lables 
            X_test   (numpy.ndarray):    test examples
            
            min_leaf_size (int):         minimum number of examples inside a node to split it
            max_depth (int):             maximum tree depth
            max_nodes (int):             maximum number of nodes
    """
    
    def __init__(self, min_leaf_size=-1, max_depth=-1, max_nodes=-1) -> None:
        self.min_leaf_size = min_leaf_size
        self.max_depth = max_depth
        self.max_nodes = max_nodes
        self.root = None
        
    def fit(self, X_train, y_train, min_leaf_size=-1, max_depth=-1, max_nodes=-1) -> None:
        """
        Builds the decision tree based on training set and hyperparameters
        """
        
        
        # update hyperparameters
        self.min_leaf_size = min_leaf_size
        self.max_depth = max_depth
        self.max_nodes = max_nodes
        
        self.root = DecisionTreeNode(X_train, y_train)
        
        # define a queue of nodes to split
        queue = [self.root]
        while queue:
            current = queue.pop(0)
    
            # check if current node can be split
            check_depth = current.depth() <= max_depth
            check_size = current.size >= min_leaf_size
            check_node_count = current.count_nodes() < max_nodes

            can_split = check_depth and check_size and check_node_count
            
            current.find_split()
            if can_split:
                current.split()
                queue.extend(current.split())
    
    def predict(self, X_test):
        """
        Predicts the class of each test point by navigating the decision tree 
        """
        if not self.root:
            raise Exception("Cannot predict before fitting the DecisionTreeClassifier")
        
        m= X_test.shape[0]
        y_pred = np.zeros(m)
        for i in range(m):
            # predict each test point's label  
            current = self.root
            while current:
                f = current.split_feat
                t = current.split_thrs
                
                if X_test[i, f] <= t:
                    if not current.lc:
                        # leaf reached, perform a majority vote to find the prediction y_pred_i of X_test_i
                        y_pred[i] = majority_vote(current.y_train)
                    current = current.lc
                else:
                    if not current.rc:
                        # leaf reached, perform a majority vote to find the prediction y_pred_i of X_test_i
                        y_pred[i] = majority_vote(current.y_train)
                    current = current.rc    
        return y_pred